# ShiftBench offline suite (real SO-101 video, no hardware)

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

* **A, policy shift:** world model calibrated on human teleop, tested on 7 different policies' rollouts.
* **B, image shift:** the same rollouts under dim/warm light, a camera bump, an occlusion and blur.
* 5 vision models (V-JEPA 2, DINOv2, SigLIP 2, CLIP, ResNet-18) × 3 world-model variants.

Results go to Google Drive: `shiftbench_results/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/kishoren18-hue/shiftbench.git 2>/dev/null; cd shiftbench && git pull -q
%cd /content/shiftbench
!pip -q install -e . av "transformers>=4.53" 2>&1 | tail -1
import torch; print("CUDA:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
OUT = "/content/drive/MyDrive/shiftbench_results/armnet_block_stack"
!python scripts/offline_suite.py --out {OUT} --dry_run

In [ ]:
# Launch in the background so a dropped browser connection doesn't kill it
import subprocess
job = subprocess.Popen(f"python scripts/offline_suite.py --out {OUT} > /content/suite.log 2>&1", shell=True)
print("started, pid", job.pid)

In [ ]:
# Re-run this cell to check progress
print("running" if job.poll() is None else f"finished, exit code {job.poll()}")
!tail -n 30 /content/suite.log

In [ ]:
# After the suite finishes: replay demo videos (camera bump on real footage)
!python scripts/replay_demo.py --detector {OUT}/detector_vjepa2 --encoder vjepa2 \
    --traces {OUT}/traces.parquet --out {OUT}/demo --n 3
!ls -la {OUT}/demo